<a href="https://colab.research.google.com/github/wiwillym/nlp-ku-project/blob/error-analysis-z/notebooks/8_zero_shot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 43 extension: zero-shot Telugu QA

**Hypothesis (stated before running).** The integrated QA model is fine-tuned on Arabic, Korean, and Telugu, while our zero-shot model is fine-tuned on Arabic and Korean only. We expect the zero-shot model to perform worse on Telugu because it has not been fine-tuned on Telugu QA examples.

- **Primary metrics:** Telugu Exact Match (EM) and token-level F1, compared with the integrated QA model on the same Telugu validation examples.
- **Expected direction:** zero-shot model < core model on both EM and token-level F1.
- **Falsified if:** the zero-shot model performs equally well or better on either primary metric. We will report the observed differences rather than assume the hypothesis is correct.
- **Secondary metric:** answerability macro-F1, to examine how well each model distinguishes answerable from unanswerable questions.

**Protocol:** Fine-tune `xlm-roberta-base` on Arabic and Korean training examples only. Use Arabic and Korean validation data for early stopping and model selection. Evaluate the selected model on the Telugu validation set, which is excluded from training and model selection. Compare it with the core integrated QA model using the same Telugu examples, answer-span evaluation procedure, and no-answer thresholding protocol. Report the scores and absolute differences. The challenge set is not used for this experiment.

In [45]:
import os
import sys
import random
import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    get_linear_schedule_with_warmup,
)

from src.data import load_tokenised_data
from src.integrated_qa import (
    build_qa_features,
    QATrainDataset,
    train_collate_fn,
    val_collate_fn,
    f1_score_squad,
    exact_match_score,
)
from src.utils import enforce_reproducibility


In [40]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

enforce_reproducibility()

In [41]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 384
DOC_STRIDE = 128

SAVE_PATH = "../checkpoints/zero_shot_telugu_qa"

In [42]:
df_train, df_val = load_tokenised_data()

train_source = df_train[
    df_train["lang"].isin(["ar", "ko"])
].copy()

val_source = df_val[
    df_val["lang"].isin(["ar", "ko"])
].copy()

test_telugu = df_val[
    df_val["lang"] == "te"
].copy()

print("Training examples:", len(train_source))
print("Arabic/Korean validation examples:", len(val_source))
print("Telugu evaluation examples:", len(test_telugu))
print("\nTraining distribution:")
print(train_source["lang"].value_counts())

assert set(train_source["lang"].unique()) == {"ar", "ko"}
assert "te" not in set(val_source["lang"].unique())
assert "te" not in set(train_source["lang"].unique())

Training examples: 4980
Arabic/Korean validation examples: 771
Telugu evaluation examples: 384

Training distribution:
lang
ar    2558
ko    2422
Name: count, dtype: int64


In [43]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",
)

# Freeze the encoder first
for param in model.roberta.parameters():
    param.requires_grad = False

# Unfreeze the last four transformer layers
for layer in model.roberta.encoder.layer[-4:]:
    for param in layer.parameters():
        param.requires_grad = True

# Train the QA output layer
for param in model.qa_outputs.parameters():
    param.requires_grad = True

model = model.to(device)

print("Model initialized.")
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad),
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForQuestionAnswering LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
qa_outputs.weight           | MISSING    | 
qa_outputs.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model initialized.
Trainable parameters: 28353026


In [44]:
train_features = build_qa_features(
    train_source,
    tokenizer,
    max_length=MAX_LENGTH,
    doc_stride=DOC_STRIDE,
)

val_features = build_qa_features(
    val_source,
    tokenizer,
    max_length=MAX_LENGTH,
    doc_stride=DOC_STRIDE,
)

telugu_features = build_qa_features(
    test_telugu,
    tokenizer,
    max_length=MAX_LENGTH,
    doc_stride=DOC_STRIDE,
)

train_dataset = QATrainDataset(train_features)

train_dl = DataLoader(
    train_dataset,
    collate_fn=train_collate_fn,
    shuffle=True,
    batch_size=4,
)

print("Training feature windows:", len(train_features))
print("Validation feature windows:", len(val_features))
print("Telugu feature windows:", len(telugu_features))
print("Training batches per epoch:", len(train_dl))

Training feature windows: 5144
Validation feature windows: 796
Telugu feature windows: 390
Training batches per epoch: 1286


In [46]:
def qa_loss_collate_fn(batch):
    keys = [
        "input_ids",
        "attention_mask",
        "start_positions",
        "end_positions",
    ]
    return {
        key: torch.tensor(
            [item[key] for item in batch],
            dtype=torch.long,
        )
        for key in keys
    }

val_loss_dl = DataLoader(
    val_features,
    collate_fn=qa_loss_collate_fn,
    batch_size=32,
    shuffle=False,
)

print("Validation batches:", len(val_loss_dl))


Validation batches: 25


In [47]:

LR = 2e-5
N_EPOCHS = 20
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 200

no_decay = ["bias", "LayerNorm.weight"]

optimizer_grouped_parameters = [
    {
        "params": [
            p for name, p in model.named_parameters()
            if p.requires_grad
            and not any(nd in name for nd in no_decay)
        ],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [
            p for name, p in model.named_parameters()
            if p.requires_grad
            and any(nd in name for nd in no_decay)
        ],
        "weight_decay": 0.0,
    },
]

optimizer = AdamW(
    optimizer_grouped_parameters,
    lr=LR,
)

total_training_steps = N_EPOCHS * len(train_dl)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_training_steps,
)

print("Total planned training steps:", total_training_steps)
print("Learning rate:", LR)
print("Maximum epochs:", N_EPOCHS)


Total planned training steps: 25720
Learning rate: 2e-05
Maximum epochs: 20


In [49]:

def evaluate_validation_loss(model, val_loss_dl, device):
    model.eval()

    total_loss = 0.0
    total_windows = 0

    with torch.inference_mode():
        for batch in tqdm(
            val_loss_dl,
            desc="Validation loss",
            leave=False,
        ):
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)

            n = batch["input_ids"].shape[0]
            total_loss += outputs.loss.item() * n
            total_windows += n

    return total_loss / total_windows

In [50]:
def train(
    model,
    train_dl,
    val_loss_dl,
    optimizer,
    scheduler,
    n_epochs,
    device,
    checkpoint_path,
    patience=3,
):
    history = []
    best_val_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0

    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

    for epoch in range(n_epochs):
        model.train()
        train_losses = []

        for batch in tqdm(
            train_dl,
            desc=f"Epoch {epoch + 1}/{n_epochs}",
        ):
            optimizer.zero_grad()

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            scheduler.step()

            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_loss = evaluate_validation_loss(
            model,
            val_loss_dl,
            device,
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
        })

        print(
            f"Epoch {epoch + 1}: "
            f"train_loss={train_loss:.4f}, "
            f"val_loss={val_loss:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            epochs_without_improvement = 0

            torch.save(
                model.state_dict(),
                checkpoint_path,
            )
            print("Best checkpoint saved.")
        else:
            epochs_without_improvement += 1
            print(
                "No improvement for "
                f"{epochs_without_improvement} epoch(s)."
            )

            if epochs_without_improvement >= patience:
                print("Early stopping.")
                break

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True,
        )
    )
    model.eval()

    print(f"Best epoch: {best_epoch}")
    print(f"Best validation loss: {best_val_loss:.4f}")

    return pd.DataFrame(history)


In [51]:

checkpoint_path = os.path.join(
    SAVE_PATH,
    "best_state_dict.pt",
)

history = train(
    model=model,
    train_dl=train_dl,
    val_loss_dl=val_loss_dl,
    optimizer=optimizer,
    scheduler=scheduler,
    n_epochs=N_EPOCHS,
    device=device,
    checkpoint_path=checkpoint_path,
    patience=3,
)

history


Epoch 1/20: 100%|██████████| 1286/1286 [02:29<00:00,  8.59it/s]


Epoch 1: train_loss=3.0490, val_loss=2.4096
Best checkpoint saved.


Epoch 2/20: 100%|██████████| 1286/1286 [02:26<00:00,  8.79it/s]


Epoch 2: train_loss=2.1009, val_loss=1.8439
Best checkpoint saved.


Epoch 3/20: 100%|██████████| 1286/1286 [02:26<00:00,  8.79it/s]


...

Epoch 6/20: 100%|██████████| 1286/1286 [02:25<00:00,  8.82it/s]


Epoch 6: train_loss=1.0154, val_loss=1.8098
No improvement for 2 epoch(s).


Epoch 7/20: 100%|██████████| 1286/1286 [02:24<00:00,  8.90it/s]


Epoch 7: train_loss=0.8275, val_loss=1.9797
No improvement for 3 epoch(s).
Early stopping.
Best epoch: 4
Best validation loss: 1.6354


,epoch,train_loss,val_loss
0,1,3.048979,2.409642
1,2,2.100880,1.843896
2,3,1.728693,1.725146
3,4,1.482418,1.635363
4,5,1.235586,1.713670
5,6,1.015443,1.809780
6,7,0.827543,1.979699


In [52]:
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

history.to_csv(
    os.path.join(SAVE_PATH, "training_history.csv"),
    index=False,
)

print("Model and tokenizer saved to:", SAVE_PATH)
print("Best checkpoint:", checkpoint_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to: ../checkpoints/zero_shot_telugu_qa
Best checkpoint: ../checkpoints/zero_shot_telugu_qa/best_state_dict.pt


In [55]:
telugu_prediction_features = [
    {
        "input_ids": feature["input_ids"],
        "attention_mask": feature["attention_mask"],
        "example_index": feature["example_index"],
        "offset_mapping": [
            offset if seq_id == 1 else None
            for offset, seq_id in zip(
                feature["offset_mapping"],
                feature["sequence_ids"],
            )
        ],
    }
    for feature in telugu_features
]

telugu_dl = DataLoader(
    telugu_prediction_features,
    collate_fn=val_collate_fn,
    batch_size=32,
)

print("Telugu examples:", len(test_telugu))
print("Telugu feature windows:", len(telugu_prediction_features))
print("Telugu prediction batches:", len(telugu_dl))


Telugu examples: 384
Telugu feature windows: 390
Telugu prediction batches: 13


In [56]:
def predict(model, data_loader, device):
    model.eval()

    start_logits_all = []
    end_logits_all = []

    with torch.inference_mode():
        for batch in tqdm(data_loader, desc="Predicting Telugu"):
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )

            start_logits_all.extend(
                outputs.start_logits.cpu().numpy()
            )
            end_logits_all.extend(
                outputs.end_logits.cpu().numpy()
            )

    return start_logits_all, end_logits_all


telugu_logits = predict(model, telugu_dl, device)

print("Number of start-logit arrays:", len(telugu_logits[0]))
print("Number of end-logit arrays:", len(telugu_logits[1]))
print("Tokens in the first window:", len(telugu_logits[0][0]))

Predicting Telugu: 100%|██████████| 13/13 [00:07<00:00,  1.85it/s]

Number of start-logit arrays: 390
Number of end-logit arrays: 390
Tokens in the first window: 274


In [57]:
from collections import defaultdict

def post_process_predictions(
    examples,
    features,
    logits,
    threshold=0.0,
    num_possible_answers=20,
    max_answer_length=30,
):
    start_logits_all, end_logits_all = logits

    features_per_example = defaultdict(list)

    for i, feature in enumerate(features):
        features_per_example[feature["example_index"]].append(i)

    predictions = []

    for example_idx, sample in examples.reset_index(drop=True).iterrows():
        feature_indices = features_per_example[example_idx]
        context = sample["context"]

        best_span_score = -np.inf
        best_span_text = ""
        best_null_score = np.inf

        for feature_idx in feature_indices:
            start_logits = start_logits_all[feature_idx]
            end_logits = end_logits_all[feature_idx]
            offset_mapping = features[feature_idx]["offset_mapping"]

            # The CLS token represents the no-answer option.
            null_score = start_logits[0] + end_logits[0]
            best_null_score = min(best_null_score, null_score)

            start_indices = np.argsort(start_logits)[::-1][:num_possible_answers]
            end_indices = np.argsort(end_logits)[::-1][:num_possible_answers]

            for start_index in start_indices:
                for end_index in end_indices:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue

                    if (
                        start_index > end_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    score = (
                        start_logits[start_index]
                        + end_logits[end_index]
                    )

                    if score > best_span_score:
                        start_char = offset_mapping[start_index][0]
                        end_char = offset_mapping[end_index][1]

                        best_span_score = score
                        best_span_text = context[start_char:end_char]

        no_answer_score = best_null_score - best_span_score
        prediction = (
            "" if no_answer_score > threshold else best_span_text
        )

        predictions.append({
            "example_index": example_idx,
            "prediction": prediction,
            "no_answer_score": no_answer_score,
        })

    return pd.DataFrame(predictions)


telugu_predictions = post_process_predictions(
    test_telugu,
    telugu_prediction_features,
    telugu_logits,
    threshold=np.inf,
)

print("Number of Telugu predictions:", len(telugu_predictions))
display(telugu_predictions.head())


Number of Telugu predictions: 384


,example_index,prediction,no_answer_score
0,0,Portland,-13.519731
1,1,Indonesia,-8.108157
2,2,Russia,-6.412968
3,3,1914,-5.985926
4,4,28 July 1914,-9.187481


In [58]:
source_prediction_features = [
    {
        "input_ids": feature["input_ids"],
        "attention_mask": feature["attention_mask"],
        "example_index": feature["example_index"],
        "offset_mapping": [
            offset if seq_id == 1 else None
            for offset, seq_id in zip(
                feature["offset_mapping"],
                feature["sequence_ids"],
            )
        ],
    }
    for feature in val_features
]

source_dl = DataLoader(
    source_prediction_features,
    collate_fn=val_collate_fn,
    batch_size=32,
)

print("Arabic/Korean validation examples:", len(val_source))
print("Arabic/Korean feature windows:", len(source_prediction_features))
print("Prediction batches:", len(source_dl))


Arabic/Korean validation examples: 771
Arabic/Korean feature windows: 796
Prediction batches: 25
